# Dealing with Lookahead Conflicts

This notebook discusses conflicts that have their origin in insufficient lookahead.
We will discuss the following grammar:
```
    a : b "U" "V"
      | c "U" "W"

    b : "X"

    c : "X"
```
This grammar is *unambiguous*:  There are only two string that can be derived from the start symbol `a`:
 * `"XUV"`,
 * `"XUW"`.

Each of these strings has exactly one parse tree.
Nevertheless, the grammar does not have the `LR(1)` property and is therefore certainly not an `LALR(1)` grammar.

## Specification of the Grammar

Below is the grammar from above written in the format for `Larkita`.

In [ ]:
from IPython.display import Code

Code(filename='look-ahead.lark', language='larkita')

`Larkita` always writes a description of the states of the LR(1) automaton into an output file.  It does so even if
it refuses to build a parser.

In [ ]:
from larkita import Larkita, GrammarError

## Trying to Build an `LR(1)` Parser

When we try to build a parser for a grammar that contains a *reduce/reduce* conflict, `Larkita` raises an exception
of class `GrammarError`.  In contrast to a *shift/reduce* conflict, which is resolved in favour of shifting, there is
no default resolution for a *reduce/reduce* conflict.  (The only exception are rules with different priorities.)

In [ ]:
try:
    Larkita.open('look-ahead.lark', out_file='look-ahead.out')
    print('No conflict.')
except GrammarError as e:
    print(e)

## Why Is There a Conflict?

If we compute the `LR` states of this grammar, we find, among others, the state
$$ \{\; b \rightarrow \texttt{'X'} \bullet : \texttt{'U'}, \quad
       c \rightarrow \texttt{'X'} \bullet : \texttt{'U'} \;\}, $$
which is exactly the state named in the error message above.  Since the set of follow tokens is the same for
both rules, we have a *reduce/reduce* conflict here.

The conflict is caused by the fact that the parser, with a lookahead of only *one* token, cannot decide
whether an `'X'` should be interpreted as a `b` or as a `c`, because this is only decided when the character
*following* the `'U'` is read:

* if this is a `'V'`, then the rule $a \rightarrow b\,\texttt{'U'}\,\texttt{'V'}$ is used and consequently
  the `'X'` has to be interpreted as a `b`;
* if, however, the second token after the `'X'` is a `'W'`, then the rule
  $a \rightarrow c\,\texttt{'U'}\,\texttt{'W'}$ is used and consequently the `'X'` is to be read as a `c`.

Although no parser has been created, we can inspect the states, as `Larkita` has written them into the file
`look-ahead.out` before raising the exception.  The conflict is listed below the actions of the offending state.

In [ ]:
with open('look-ahead.out') as handle:
    print(handle.read())

## The Real Solution

In contrast to `Lark`, `Larkita` offers no *Earley* parser as an escape hatch.  This is no loss, since switching to an
`Earley` parser would hide the problem instead of solving it.  The grammar can be rewritten so that it becomes an
`LR(1)` grammar:  if the context `'U'`, which follows both `b` and `c`, is pulled into the rules for `b` and `c`,
then the conflict disappears, because the state in which the conflict occurred now has the form
$$ \{\; b \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'V'}, \quad
       c \rightarrow \texttt{'X'}\,\texttt{'U'} \bullet : \texttt{'W'} \;\} $$
and here the next token decides which rule to reduce with.  This is done in the notebook
[04-Look-Ahead-Solved.ipynb](04-Look-Ahead-Solved.ipynb).